# Retrieval Evaluation Notebook (dedicated, metrics-only)

This is a **dedicated retrieval-evaluation notebook**, distinct from [`notebooks/faiss_retrieval_ready.ipynb`](./faiss_retrieval_ready.ipynb) (which is a demonstration notebook for the vector + hybrid pipeline, not an evaluation harness).

It runs the frozen QA benchmark through:

1. **Vector-only retrieval** (US1) — [`VectorRetriever`](../src/retrieval/retriever.py) with `graph_expansion=None`.
2. **Hybrid retrieval** (US2) — the primary `GRAPH_MODULE §10` sequence: unfiltered vector pre-pass → traversal starts → graph traversal → whitelist → graph-guided filtered vector search → `GraphExpansion` → fusion.
3. **Side-by-side comparison** (US3) of the same metrics at the same `k` cutoffs.
4. **Persisted artifacts** (US4) — per-case JSONL, aggregate metrics JSON, and self-contained markdown reports for each mode plus the comparison.

All metrics come exclusively from [`src/evaluation/metrics.py`](../src/evaluation/metrics.py) via [`src/evaluation/retrieval_eval_report.py`](../src/evaluation/retrieval_eval_report.py). This notebook contains **no** exact-match / token-F1 / ROUGE-L / judge-score generation metrics (FR-018) — those belong to `scripts/evaluate_e2e.py`.

## Section outline

```text
1. Environment setup
2. Import surface check
3. Config
4. Preflight (QA path, FAISS artifacts, graph source)
5. Vector-only evaluation (US1)
6. Hybrid evaluation (US2)
7. Comparison (US3)
8. Persist artifacts (US4)
```

See [`specs/006-retrieval-eval-notebook/quickstart.md`](../specs/006-retrieval-eval-notebook/quickstart.md) for the full operator guide and validation scenarios V1-V8.

**This notebook does not modify** `data/benchmark/qa_final.jsonl` / `data/qa_final.jsonl` (read-only), nor `scripts/evaluate_retrieval.py`, `scripts/evaluate_e2e.py`, `src/evaluation/retriever_factory.py`, or `notebooks/faiss_retrieval_ready.ipynb`.


## 1. Environment setup


In [ ]:
# Optional: install runtime dependencies if your environment does not have them yet.
# Uncomment and run once if needed.
# %pip install -q faiss-cpu sentence-transformers pandas


In [ ]:
from pathlib import Path
import json
import logging
import os
import sys
import time

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # Useful if the notebook is launched from notebooks/ (FR-016 path portability).
    PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print('HF Hub token detected in environment.')
else:
    logging.getLogger('huggingface_hub.utils._http').setLevel(logging.ERROR)
    print('HF_TOKEN not set; suppressing the Hugging Face unauthenticated-request warning.')

print('Project root:', PROJECT_ROOT)
print('src on path:', SRC_DIR.exists())


## 2. Import surface check

Confirms every module this notebook depends on (evaluation helpers, vector retrieval stack, knowledge graph package) is importable before any config/preflight logic runs.


In [ ]:
# ### IMPORT_SURFACE_CHECK (T002)
from evaluation.eligibility import EligibleCase, EligibilitySummary, select_eligible_cases
from evaluation.hybrid_fusion import (
    HybridFusionResult,
    TraversalStartSet,
    build_traversal_starts,
    fuse_hybrid_chunk_ids,
)
from evaluation.io_utils import read_jsonl, write_json, write_jsonl
from evaluation.metrics import aggregate, aggregate_by
from evaluation.retrieval_eval_report import (
    ComparisonSummary,
    HybridDiagnostics,
    ModeRunSummary,
    RetrievalCaseResult,
    RetrievalMode,
    build_case_metrics_row,
    build_comparison,
    metric_keys_for,
    write_case_jsonl,
    write_comparison_report,
    write_markdown_report,
    write_metrics_json,
)

from retrieval.config import VectorIndexConfig
from retrieval.embeddings import SentenceTransformerEmbedder
from retrieval.retriever import VectorRetriever
from retrieval.schema import RetrievalResult, RetrievedChunk
from retrieval.sqlite_faiss_store import SQLitePayloadFaissVectorStore

from knowledge_graph.context_schema import GraphGuidedFilter
from knowledge_graph.expansion import GraphExpansion
from knowledge_graph.facade import KnowledgeGraphFacade
from knowledge_graph.loader import GraphLoaderPaths
from knowledge_graph.persist import load_knowledge_graph
from knowledge_graph.traversal import GraphTraversal, TraversalMode, TraversalResult

print('Import surface OK: evaluation, retrieval, and knowledge_graph modules all importable.')


## 3. Config

Mirrors [`quickstart.md`](../specs/006-retrieval-eval-notebook/quickstart.md)'s config cell. Paths are resolved relative to `PROJECT_ROOT` for portability (FR-016).


In [ ]:
# === Config (near top per quickstart.md) ===

# --- Benchmark + output ---
QA_PATH = PROJECT_ROOT / 'data' / 'benchmark' / 'qa_final.jsonl'  # default; falls back per research R10
OUT_DIR = PROJECT_ROOT / 'evaluation_runs' / 'retrieval_notebook' / 'run1'
TOP_K_LIST = [1, 5, 10]
SAMPLE_LIMIT = 20  # None for full benchmark

# --- Vector retrieval ---
FILTER_PROFILE = 'current_law'
SCORE_THRESHOLD = None
TOP_K_RETRIEVE = 20
TOP_N = 10

INDEX_DIR = PROJECT_ROOT / 'data' / 'faiss_index'
EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'  # must match the built index

RUN_VECTOR_ONLY = True
RUN_HYBRID = True

# --- Hybrid / graph ---
GRAPH_PICKLE_PATH = PROJECT_ROOT / 'data' / 'graph' / 'knowledge_graph.gpickle'
V2_DATA_DIR = PROJECT_ROOT / 'data' / 'v2'
ALLOW_JSONL_GRAPH_REBUILD = False

TRAVERSAL_MODE = 'basis'
TRAVERSAL_MAX_DEPTH = 3
PREPASS_TOP_N = 10
MAX_TRAVERSAL_STARTS = 5

HYBRID_MAX_HOP = 2
HYBRID_MAX_CONTEXT = 20

AS_OF_DATE = None
LOCAL_EXPAND_UNITS = False  # keep False for official scored hybrid runs

RUN_CONFIG = {
    'qa_path': str(QA_PATH),
    'out_dir': str(OUT_DIR),
    'top_k_list': TOP_K_LIST,
    'sample_limit': SAMPLE_LIMIT,
    'filter_profile': FILTER_PROFILE,
    'score_threshold': SCORE_THRESHOLD,
    'top_k_retrieve': TOP_K_RETRIEVE,
    'top_n': TOP_N,
    'index_dir': str(INDEX_DIR),
    'embedding_model': EMBEDDING_MODEL,
    'run_vector_only': RUN_VECTOR_ONLY,
    'run_hybrid': RUN_HYBRID,
    'graph_pickle_path': str(GRAPH_PICKLE_PATH),
    'v2_data_dir': str(V2_DATA_DIR),
    'allow_jsonl_graph_rebuild': ALLOW_JSONL_GRAPH_REBUILD,
    'traversal_mode': TRAVERSAL_MODE,
    'traversal_max_depth': TRAVERSAL_MAX_DEPTH,
    'prepass_top_n': PREPASS_TOP_N,
    'max_traversal_starts': MAX_TRAVERSAL_STARTS,
    'hybrid_max_hop': HYBRID_MAX_HOP,
    'hybrid_max_context': HYBRID_MAX_CONTEXT,
    'as_of_date': AS_OF_DATE,
    'local_expand_units': LOCAL_EXPAND_UNITS,
}

print(json.dumps(RUN_CONFIG, indent=2))


## 4. Preflight

Resolves the QA benchmark path (research R10 fallback), confirms required FAISS artifacts exist, and confirms the graph source mode for hybrid (pickle preferred; JSONL rebuild opt-in only). Stops with a clear error for QA/FAISS; marks `hybrid_available=False` (never silently) if the graph is unavailable (FR-011).


In [ ]:
# ### PREFLIGHT — QA benchmark path (research R10: no silent default swap without a warning)
_qa_path_overridden = 'QA_PATH' in dir() and QA_PATH != PROJECT_ROOT / 'data' / 'benchmark' / 'qa_final.jsonl'

if not QA_PATH.exists() and not _qa_path_overridden:
    _fallback_qa_path = PROJECT_ROOT / 'data' / 'qa_final.jsonl'
    if _fallback_qa_path.exists():
        print(
            f'WARNING: default QA_PATH {QA_PATH} not found; '
            f'falling back to {_fallback_qa_path} (research.md R10). '
            'Set QA_PATH explicitly to silence this warning.'
        )
        QA_PATH = _fallback_qa_path
        RUN_CONFIG['qa_path'] = str(QA_PATH)

if not QA_PATH.exists():
    raise FileNotFoundError(
        f'QA benchmark not found at {QA_PATH}. Set QA_PATH to the frozen benchmark file before proceeding.'
    )

print('QA_PATH resolved:', QA_PATH)

qa_rows = list(read_jsonl(QA_PATH))
print(f'Loaded {len(qa_rows):,} QA benchmark rows (read-only; never modified — FR-017).')


In [ ]:
# ### PREFLIGHT — FAISS artifacts (required for both vector-only and hybrid)
_required_faiss_files = [INDEX_DIR / 'index.faiss', INDEX_DIR / 'payloads.jsonl']
_missing_faiss = [p for p in _required_faiss_files if not p.exists()]

if _missing_faiss:
    for p in _missing_faiss:
        print('MISSING:', p)
    raise FileNotFoundError(
        f'Required FAISS artifacts missing under {INDEX_DIR}. '
        'Build the index (scripts/build_vector_index.py) or point INDEX_DIR at an existing pack.'
    )

print('Required FAISS artifacts found under', INDEX_DIR)
for p in _required_faiss_files:
    print(f'  {p.name}: {p.stat().st_size / 1024 / 1024:.2f} MB')


In [ ]:
# ### PREFLIGHT — graph source mode for hybrid (FR-011: never silently fall back under a hybrid label)
hybrid_available = True
hybrid_unavailable_reason = None  # one of: graph_unavailable | traversal_unavailable | expansion_unavailable

kg_graph = None
kg_facade = None
kg_traversal = None
graph_expansion = None

if not RUN_HYBRID:
    hybrid_available = False
    hybrid_unavailable_reason = 'graph_unavailable'
    print('RUN_HYBRID=False; hybrid evaluation will be skipped.')
else:
    _pickle_ready = GRAPH_PICKLE_PATH.exists()
    _jsonl_paths = GraphLoaderPaths(data_dir=V2_DATA_DIR)
    try:
        _jsonl_ready = not bool([p for p in _jsonl_paths.required_paths() if not p.exists()])
    except Exception:
        _jsonl_ready = False

    if _pickle_ready:
        try:
            print('Loading structural graph from pickle:', GRAPH_PICKLE_PATH)
            _graph_load_t0 = time.perf_counter()
            kg_pickle_result = load_knowledge_graph(GRAPH_PICKLE_PATH)
            kg_graph = kg_pickle_result.graph
            print(f'Graph loaded from pickle in {time.perf_counter() - _graph_load_t0:.2f}s')
            if kg_pickle_result.warnings:
                print('Load warnings:')
                for w in list(kg_pickle_result.warnings)[:20]:
                    print(' -', w)
        except Exception as exc:
            hybrid_available = False
            hybrid_unavailable_reason = 'graph_unavailable'
            print('Graph pickle load FAILED:', exc)
    elif ALLOW_JSONL_GRAPH_REBUILD and _jsonl_ready:
        try:
            print('Rebuilding structural graph from JSONL sources (opt-in):', V2_DATA_DIR)
            kg_facade = KnowledgeGraphFacade(paths=_jsonl_paths)
            _graph_build_t0 = time.perf_counter()
            kg_build_result = kg_facade.build_graph()
            kg_graph = kg_build_result.graph
            print(f'Graph built from JSONL in {time.perf_counter() - _graph_build_t0:.2f}s')
        except Exception as exc:
            hybrid_available = False
            hybrid_unavailable_reason = 'graph_unavailable'
            print('Graph JSONL rebuild FAILED:', exc)
    else:
        hybrid_available = False
        hybrid_unavailable_reason = 'graph_unavailable'
        print(f'Graph pickle not found at {GRAPH_PICKLE_PATH} and JSONL rebuild is not available/enabled.')
        print('hybrid_available=False (FR-011). Vector-only evaluation still runs normally.')

    if kg_graph is not None:
        try:
            kg_facade = kg_facade or KnowledgeGraphFacade(paths=_jsonl_paths)
            kg_traversal = kg_facade.build_traversal(kg_graph)
        except Exception as exc:
            hybrid_available = False
            hybrid_unavailable_reason = 'traversal_unavailable'
            print('GraphTraversal construction FAILED:', exc)

    if kg_graph is not None and hybrid_available:
        try:
            graph_expansion = GraphExpansion(kg_graph)
        except Exception as exc:
            hybrid_available = False
            hybrid_unavailable_reason = 'expansion_unavailable'
            print('GraphExpansion construction FAILED:', exc)

print()
print('hybrid_available:', hybrid_available)
print('hybrid_unavailable_reason:', hybrid_unavailable_reason)


## 5. Vector-only evaluation (US1)

Loads the FAISS store + embedder, builds a `VectorRetriever` with `graph_expansion=None`, evaluates every eligible case, and aggregates metrics via `evaluation.metrics`.


In [ ]:
# ### US1 — eligibility (never scores unanswerable / missing-ground-truth rows)
eligibility_summary = select_eligible_cases(qa_rows, SAMPLE_LIMIT)

print('total_rows examined:', eligibility_summary.total_rows)
print('eligible:', len(eligibility_summary.eligible))
print('skipped_unanswerable:', eligibility_summary.skipped_unanswerable)
print('skipped_missing_ground_truth:', eligibility_summary.skipped_missing_ground_truth)


In [ ]:
# ### US1 — load FAISS store + embedder + vector-only retriever
vector_only_cases: list[RetrievalCaseResult] = []
vector_only_summary: ModeRunSummary | None = None

if RUN_VECTOR_ONLY:
    vector_config = VectorIndexConfig(
        embedding_model=EMBEDDING_MODEL,
        top_k=TOP_K_RETRIEVE,
        top_n=TOP_N,
        score_threshold=(SCORE_THRESHOLD if SCORE_THRESHOLD is not None else VectorIndexConfig().score_threshold),
        expand_units=LOCAL_EXPAND_UNITS,
    )

    vector_store = SQLitePayloadFaissVectorStore.load(INDEX_DIR)
    vector_embedder = SentenceTransformerEmbedder(
        EMBEDDING_MODEL,
        query_prefix=vector_config.query_prefix,
        passage_prefix=vector_config.passage_prefix,
    )
    vector_retriever = VectorRetriever(config=vector_config, embedder=vector_embedder, store=vector_store)

    print(f'Vector-only retriever ready. Loaded FAISS vectors: {vector_store.total_vectors:,}')
else:
    print('RUN_VECTOR_ONLY=False; skipping vector-only setup.')


In [ ]:
# ### US1 — run vector-only retrieval per eligible case (FR-015: one case error never aborts the run)
if RUN_VECTOR_ONLY:
    vector_only_cases = []
    _vector_error_count = 0

    for case in eligibility_summary.eligible:
        try:
            result: RetrievalResult = vector_retriever.retrieve(
                case.question,
                filter_profile=FILTER_PROFILE,
                top_k=TOP_K_RETRIEVE,
                top_n=TOP_N,
            )
            retrieved_chunk_ids = [chunk.chunk_id for chunk in result.chunks]
            metrics_row = build_case_metrics_row(
                retrieved_chunk_ids, case.ground_truth_chunk_ids, TOP_K_LIST
            )
            vector_only_cases.append(
                RetrievalCaseResult(
                    qa_id=case.qa_id,
                    mode='vector_only',
                    question=case.question,
                    category=case.category,
                    difficulty=case.difficulty,
                    answer_type=case.answer_type,
                    ground_truth_chunk_ids=sorted(case.ground_truth_chunk_ids),
                    retrieved_chunk_ids=retrieved_chunk_ids,
                    metrics=metrics_row,
                )
            )
        except Exception as exc:
            _vector_error_count += 1
            vector_only_cases.append(
                RetrievalCaseResult(
                    qa_id=case.qa_id,
                    mode='vector_only',
                    question=case.question,
                    category=case.category,
                    difficulty=case.difficulty,
                    answer_type=case.answer_type,
                    ground_truth_chunk_ids=sorted(case.ground_truth_chunk_ids),
                    retrieved_chunk_ids=[],
                    metrics={},
                    error=str(exc),
                )
            )

    print(f'Vector-only run complete: {len(vector_only_cases)} cases, {_vector_error_count} errors.')


In [ ]:
# ### US1 — aggregate + display vector-only metrics
if RUN_VECTOR_ONLY:
    _metric_keys = metric_keys_for(TOP_K_LIST)
    _scored_rows = [c.metrics for c in vector_only_cases if c.error is None]

    _overall = aggregate(_scored_rows, _metric_keys)
    _by_category = aggregate_by(
        [{**c.metrics, 'category': c.category} for c in vector_only_cases if c.error is None],
        'category',
        _metric_keys,
    )
    _by_difficulty = aggregate_by(
        [{**c.metrics, 'difficulty': c.difficulty} for c in vector_only_cases if c.error is None],
        'difficulty',
        _metric_keys,
    )
    _by_answer_type = aggregate_by(
        [{**c.metrics, 'answer_type': c.answer_type} for c in vector_only_cases if c.error is None],
        'answer_type',
        _metric_keys,
    )

    vector_only_summary = ModeRunSummary(
        mode='vector_only',
        config=RUN_CONFIG,
        total_rows=eligibility_summary.total_rows,
        evaluated=len(_scored_rows),
        skipped_unanswerable=eligibility_summary.skipped_unanswerable,
        skipped_missing_ground_truth=eligibility_summary.skipped_missing_ground_truth,
        error_count=sum(1 for c in vector_only_cases if c.error is not None),
        overall=_overall,
        by_category=_by_category,
        by_difficulty=_by_difficulty,
        by_answer_type=_by_answer_type,
        hybrid_available=True,
    )

    print('=== Vector-only overall metrics ===')
    for key in _metric_keys:
        print(f'  {key}: {_overall.get(key, 0.0):.4f}')
    print('evaluated:', vector_only_summary.evaluated, '| errors:', vector_only_summary.error_count)


## 6. Hybrid evaluation (US2)

Implements the primary `GRAPH_MODULE §10` sequence per case:

```text
unfiltered vector pre-pass
  -> build_traversal_starts (never reads ground_truth.*)
  -> GraphTraversal.traverse per start_id
  -> whitelist of visited id_strs
  -> graph-guided filtered vector search (id_str_filter)
  -> GraphExpansion.expand on the filtered seed chunk ids
  -> fuse_hybrid_chunk_ids (seeds -> expansion -> extra traversal chunks, keep-first dedupe)
```

Skipped entirely (with an explicit reason, never silently) when `hybrid_available=False`.


In [ ]:
# ### US2 — hybrid retriever setup (only if hybrid_available and RUN_HYBRID)
hybrid_cases: list[RetrievalCaseResult] = []
hybrid_summary: ModeRunSummary | None = None

hybrid_retriever: VectorRetriever | None = None
overlays_available = False  # optional; never required for hybrid success (research R9)

if RUN_HYBRID and hybrid_available:
    hybrid_config = VectorIndexConfig(
        embedding_model=EMBEDDING_MODEL,
        top_k=TOP_K_RETRIEVE,
        top_n=TOP_N,
        score_threshold=(SCORE_THRESHOLD if SCORE_THRESHOLD is not None else VectorIndexConfig().score_threshold),
        expand_units=LOCAL_EXPAND_UNITS,
    )

    if RUN_VECTOR_ONLY:
        # Reuse the already-loaded FAISS store + embedder to avoid a second cold load.
        hybrid_store = vector_store
        hybrid_embedder = vector_embedder
    else:
        hybrid_store = SQLitePayloadFaissVectorStore.load(INDEX_DIR)
        hybrid_embedder = SentenceTransformerEmbedder(
            EMBEDDING_MODEL,
            query_prefix=hybrid_config.query_prefix,
            passage_prefix=hybrid_config.passage_prefix,
        )

    # graph_expansion is intentionally NOT wired here: hybrid evaluation applies GraphExpansion
    # explicitly per case (below) rather than via VectorRetriever's automatic same-unit expansion,
    # so the pre-pass retriever and the filtered retriever below use graph_expansion=None.
    hybrid_retriever = VectorRetriever(config=hybrid_config, embedder=hybrid_embedder, store=hybrid_store)

    overlays_available = False
    print('Hybrid retriever ready (graph_expansion applied explicitly via GraphExpansion.expand per case).')
else:
    print('Hybrid evaluation skipped. hybrid_available:', hybrid_available, '| reason:', hybrid_unavailable_reason)


In [ ]:
# ### US2 — run hybrid retrieval per eligible case (GRAPH_MODULE §10 sequence; FR-015 per-case errors)
if RUN_HYBRID and hybrid_available:
    hybrid_cases = []
    _hybrid_error_count = 0

    for case in eligibility_summary.eligible:
        try:
            # 1. Unfiltered vector pre-pass (never reads ground_truth.*).
            prepass_result: RetrievalResult = hybrid_retriever.retrieve(
                case.question,
                filter_profile=FILTER_PROFILE,
                top_k=PREPASS_TOP_N,
                top_n=PREPASS_TOP_N,
            )
            prepass_hits = list(prepass_result.chunks)

            # 2. Map pre-pass hits to traversal starts.
            start_set = build_traversal_starts(prepass_hits, TRAVERSAL_MODE, MAX_TRAVERSAL_STARTS)

            prepass_empty_start = start_set.empty
            visited_ids: set[str] = set()
            traversal_visited_count = 0

            if not prepass_empty_start:
                # 3. Traverse the graph from each start id, collecting visited node ids.
                for start_id in start_set.start_ids:
                    traversal_result: TraversalResult = kg_traversal.traverse(
                        start_id, TRAVERSAL_MODE, max_depth=TRAVERSAL_MAX_DEPTH
                    )
                    visited_ids.update(traversal_result.visited_ids)
                traversal_visited_count = len(visited_ids)

            # 4. Whitelist for the graph-guided filter.
            whitelist_id_strs = tuple(sorted(visited_ids))
            whitelist_empty = not whitelist_id_strs

            guided_filter = GraphGuidedFilter(
                id_strs=whitelist_id_strs,
                empty_filter_warning=whitelist_empty,
                filter_profile='graph_guided',
            )

            # 5. Graph-guided filtered vector search.
            filtered_result: RetrievalResult = hybrid_retriever.retrieve(
                case.question,
                graph_guided_filter=guided_filter,
                top_k=TOP_K_RETRIEVE,
                top_n=TOP_N,
            )
            filtered_seed_chunk_ids = [chunk.chunk_id for chunk in filtered_result.chunks]

            # 6. GraphExpansion on the filtered seed chunk ids.
            expansion_added: tuple[str, ...] = ()
            expansion_seed_count = len(filtered_seed_chunk_ids)
            if filtered_seed_chunk_ids:
                expansion_result = graph_expansion.expand(
                    filtered_seed_chunk_ids,
                    max_hop=HYBRID_MAX_HOP,
                    max_context=HYBRID_MAX_CONTEXT,
                )
                expansion_added = tuple(
                    cid for cid in expansion_result.ordered_context_chunks if cid not in filtered_seed_chunk_ids
                )

            # 7. Fuse: seeds -> expansion -> extra traversal chunks, keep-first dedupe (no re-ranking).
            fusion = fuse_hybrid_chunk_ids(filtered_seed_chunk_ids, expansion_added, ())

            diagnostics = HybridDiagnostics(
                traversal_mode=TRAVERSAL_MODE,
                traversal_start_ids=start_set.start_ids,
                traversal_visited_count=traversal_visited_count,
                whitelist_id_strs=whitelist_id_strs,
                whitelist_empty=whitelist_empty,
                filtered_vector_seed_chunk_ids=tuple(filtered_seed_chunk_ids),
                expansion_seed_count=expansion_seed_count,
                expansion_added_count=len(fusion.expansion_added),
                expansion_empty_added=not fusion.expansion_added,
                extra_traversal_chunk_ids=fusion.traversal_added,
                overlays_available=overlays_available,
                prepass_empty_start=prepass_empty_start,
            )

            retrieved_chunk_ids = list(fusion.retrieved_chunk_ids)
            metrics_row = build_case_metrics_row(
                retrieved_chunk_ids, case.ground_truth_chunk_ids, TOP_K_LIST
            )

            hybrid_cases.append(
                RetrievalCaseResult(
                    qa_id=case.qa_id,
                    mode='hybrid',
                    question=case.question,
                    category=case.category,
                    difficulty=case.difficulty,
                    answer_type=case.answer_type,
                    ground_truth_chunk_ids=sorted(case.ground_truth_chunk_ids),
                    retrieved_chunk_ids=retrieved_chunk_ids,
                    metrics=metrics_row,
                    hybrid_diagnostics=diagnostics,
                )
            )
        except Exception as exc:
            _hybrid_error_count += 1
            hybrid_cases.append(
                RetrievalCaseResult(
                    qa_id=case.qa_id,
                    mode='hybrid',
                    question=case.question,
                    category=case.category,
                    difficulty=case.difficulty,
                    answer_type=case.answer_type,
                    ground_truth_chunk_ids=sorted(case.ground_truth_chunk_ids),
                    retrieved_chunk_ids=[],
                    metrics={},
                    error=str(exc),
                )
            )

    print(f'Hybrid run complete: {len(hybrid_cases)} cases, {_hybrid_error_count} errors.')


In [ ]:
# ### US2 — aggregate + display hybrid metrics (respects hybrid_available gate; FR-011/FR-019)
if RUN_HYBRID and hybrid_available:
    _metric_keys = metric_keys_for(TOP_K_LIST)
    _scored_hybrid_rows = [c.metrics for c in hybrid_cases if c.error is None]

    _overall_h = aggregate(_scored_hybrid_rows, _metric_keys)
    _by_category_h = aggregate_by(
        [{**c.metrics, 'category': c.category} for c in hybrid_cases if c.error is None],
        'category',
        _metric_keys,
    )
    _by_difficulty_h = aggregate_by(
        [{**c.metrics, 'difficulty': c.difficulty} for c in hybrid_cases if c.error is None],
        'difficulty',
        _metric_keys,
    )
    _by_answer_type_h = aggregate_by(
        [{**c.metrics, 'answer_type': c.answer_type} for c in hybrid_cases if c.error is None],
        'answer_type',
        _metric_keys,
    )

    hybrid_summary = ModeRunSummary(
        mode='hybrid',
        config=RUN_CONFIG,
        total_rows=eligibility_summary.total_rows,
        evaluated=len(_scored_hybrid_rows),
        skipped_unanswerable=eligibility_summary.skipped_unanswerable,
        skipped_missing_ground_truth=eligibility_summary.skipped_missing_ground_truth,
        error_count=sum(1 for c in hybrid_cases if c.error is not None),
        overall=_overall_h,
        by_category=_by_category_h,
        by_difficulty=_by_difficulty_h,
        by_answer_type=_by_answer_type_h,
        hybrid_available=True,
    )

    print('=== Hybrid overall metrics ===')
    for key in _metric_keys:
        print(f'  {key}: {_overall_h.get(key, 0.0):.4f}')
    print('evaluated:', hybrid_summary.evaluated, '| errors:', hybrid_summary.error_count)
else:
    hybrid_summary = ModeRunSummary(
        mode='hybrid',
        config=RUN_CONFIG,
        total_rows=eligibility_summary.total_rows,
        evaluated=0,
        skipped_unanswerable=eligibility_summary.skipped_unanswerable,
        skipped_missing_ground_truth=eligibility_summary.skipped_missing_ground_truth,
        error_count=0,
        overall={},
        by_category={},
        by_difficulty={},
        by_answer_type={},
        hybrid_available=False,
        hybrid_unavailable_reason=hybrid_unavailable_reason,
    )
    print('Hybrid unavailable:', hybrid_unavailable_reason, '- vector-only results stand alone.')


## 7. Comparison (US3)

Builds a side-by-side table at each configured `metric@k` using [`build_comparison`](../src/evaluation/retrieval_eval_report.py). Shows an explicit "hybrid unavailable" note rather than fabricating hybrid numbers when hybrid did not run (FR-019, SC-007).


In [ ]:
# ### US3 — comparison
comparison = build_comparison(vector_only_summary, hybrid_summary, TOP_K_LIST)

print('hybrid_available (comparison):', comparison.hybrid_available)
print()
print(f"{'metric':<12} {'vector_only':>12} {'hybrid':>12}")
for row in comparison.rows:
    v = f"{row['vector_only']:.4f}" if row['vector_only'] is not None else 'n/a'
    h = f"{row['hybrid']:.4f}" if row['hybrid'] is not None else 'n/a'
    print(f"{row['metric']:<12} {v:>12} {h:>12}")


## 8. Persist artifacts (US4)

Writes per-case JSONL, aggregate metrics JSON, and self-contained markdown reports for each mode that ran, plus the comparison report. All writes go under `OUT_DIR`; the read-only QA benchmark file is never touched.


In [ ]:
# ### US4 — write artifacts
OUT_DIR.mkdir(parents=True, exist_ok=True)
_metric_keys = metric_keys_for(TOP_K_LIST)

if RUN_VECTOR_ONLY and vector_only_summary is not None:
    write_case_jsonl(OUT_DIR / 'vector_only_cases.jsonl', vector_only_cases)
    write_metrics_json(OUT_DIR / 'vector_only_metrics.json', vector_only_summary, RUN_CONFIG)
    write_markdown_report(OUT_DIR / 'vector_only_report.md', vector_only_summary, _metric_keys, str(QA_PATH))
    print('Wrote vector_only_cases.jsonl / vector_only_metrics.json / vector_only_report.md')

if RUN_HYBRID and hybrid_summary is not None:
    write_case_jsonl(OUT_DIR / 'hybrid_cases.jsonl', hybrid_cases)
    write_metrics_json(OUT_DIR / 'hybrid_metrics.json', hybrid_summary, RUN_CONFIG)
    write_markdown_report(OUT_DIR / 'hybrid_report.md', hybrid_summary, _metric_keys, str(QA_PATH))
    print('Wrote hybrid_cases.jsonl / hybrid_metrics.json / hybrid_report.md')

write_comparison_report(OUT_DIR / 'comparison_report.md', comparison)
print('Wrote comparison_report.md')
print()
print('All artifacts written under:', OUT_DIR)
